# 7c. A classifier trained on the model's answers

The open model of notebooks 7 and 7b classified about 5,800 records, too slow to run on all 163,169. Here I train a simple
classifier on its answers (TF-IDF on pieces of two to five letters of the author and the title, and a logistic regression) and
let it label every record of the catalog in a few seconds. The classifier copies the model, errors included, and adds errors of
its own, so its counts are not the estimates. Notebook 8 uses it only as a check: corrected with the sample, it gives figures
close to the sample's, with intervals no narrower, so the published figures come from the sample alone. For that check each
sample record gets the answer of a classifier trained without it (five-fold cross-validation).

---

# 7c. Un clasificador entrenado con las respuestas del modelo

El modelo abierto de los notebooks 7 y 7b clasificó unos 5.800 registros; es demasiado lento para pasarlo por los 163.169. Acá
entreno un clasificador simple con sus respuestas (TF-IDF sobre trozos de dos a cinco letras del autor y el título, y una
regresión logística) y con él etiqueto todos los registros del catálogo en pocos segundos. El clasificador copia al modelo, con
sus errores, y suma errores propios, así que sus conteos no son las estimaciones. El notebook 8 lo usa solo como control:
corregido con la muestra, da cifras parecidas a las de la muestra, con intervalos que no son más estrechos, así que las cifras
publicadas salen solo de la muestra. Para ese control, cada registro de la muestra recibe la respuesta de un clasificador
entrenado sin él (validación cruzada en cinco partes).

In [1]:
# Celda 1
# Paths and parameters:
%cd "E:/No sincroniza/Software desarrollado/spanish-imprints-counts"
DATA = "E:/Datos/spanish_imprints"
CATALOG = DATA + "/catalog.tsv"                               # From notebook 3.
SAMPLE = "data/llm_catalog_sample.tsv"                        # From notebook 6.
SAMPLE_RESULTS = "data/llm_catalog_sample_results.tsv"        # From notebook 7.
EXTRA_SAMPLE = "data/llm_bne16_extra_sample.tsv"              # From notebook 6.
EXTRA_RESULTS = "data/llm_bne16_extra_results.tsv"            # From notebook 7b.
CATALOG_PREDICTIONS = DATA + "/ml_catalog_predictions.tsv"    # Too large for the repository, like the catalog.
SAMPLE_PREDICTIONS = "data/ml_sample_predictions.tsv"
FOLDS = 5
SEED = 7
C = 5   # Strength of the fit; 5 did better than 1 and 20 in a five-fold test on the first sample.
CATEGORIES = ["espiritual", "predicable", "historia_sacra", "teologia_moral", "catecismo",
              "teologia", "biblia", "liturgia", "gobierno_religioso", "no_religioso"]

E:\No sincroniza\Software desarrollado\spanish-imprints-counts


In [2]:
# Celda 2
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from lexicon import fold

catalog = pd.read_csv(CATALOG, sep="\t", dtype={"id": str})
sample = pd.read_csv(SAMPLE, sep="\t", dtype={"id": str})
sample = sample.merge(pd.read_csv(SAMPLE_RESULTS, sep="\t", dtype={"id": str}), on=["source", "id"])
extra = pd.read_csv(EXTRA_SAMPLE, sep="\t", dtype={"id": str})
extra = extra.merge(pd.read_csv(EXTRA_RESULTS, sep="\t", dtype={"id": str}), on=["source", "id"])
labeled = pd.concat([sample, extra], ignore_index=True)
print("Catalog:", len(catalog), "| records with the model's answer:", len(labeled))

Catalog: 163169 | records with the model's answer: 5807


## 1. What does the classifier read?

---

## 1. ¿Qué lee el clasificador?

In [3]:
# Celda 3
# What the classifier reads: author and title, folded like the lexicon ("Christiana" -> "cristiana"):
catalog["text"] = (catalog["author_text"].fillna("") + " " + catalog["title"].fillna("")).map(fold)
labeled["text"] = (labeled["author"].fillna("") + " " + labeled["title"].fillna("")).map(fold)
labeled["text"].head(3)

0     instrucion i forma para reduzir la administra...
1     serrano miguel  s juiii mui noble i mui leal ...
2     tesauro emanuele    roman manuel  fl  impreso...
Name: text, dtype: object

In [4]:
# Celda 4
# Pieces of two to five letters inside each word, weighted by how rare they are, and a logistic regression:
classifier = make_pipeline(TfidfVectorizer(analyzer="char_wb", ngram_range=(2, 5), min_df=2, sublinear_tf=True),
                           LogisticRegression(max_iter=3000, C=C))

## 2. How well does the classifier copy the model?

---

## 2. ¿Qué tan bien copia el clasificador al modelo?

In [5]:
# Celda 5
# Each sample record gets the answer of a classifier trained on the other four fifths:
folds = StratifiedKFold(n_splits=FOLDS, shuffle=True, random_state=SEED)
labeled["ml_category"] = cross_val_predict(classifier, labeled["text"], labeled["llm_category"], cv=folds)
print(f"Same answer as the model: {(labeled['ml_category'] == labeled['llm_category']).mean():.1%}")

Same answer as the model: 77.7%


In [6]:
# Celda 6
# For each category: how many records the model and the classifier put there, how many the classifier finds of the
# model's ones, and how often it is right when it chooses that category:
rows = []
for category in CATEGORIES:
    by_model = labeled["llm_category"] == category
    by_classifier = labeled["ml_category"] == category
    if by_model.sum() == 0:
        continue   # The model never chose "teologia" in the sample.
    both = (by_model & by_classifier).sum()
    rows.append({"category": category, "model": by_model.sum(), "classifier": by_classifier.sum(),
                 "found": round(both / by_model.sum(), 2), "right": round(both / by_classifier.sum(), 2)})
pd.DataFrame(rows)

,category,model,classifier,found,right
0,espiritual,982,984,0.81,0.80
1,predicable,527,537,0.84,0.83
2,historia_sacra,317,223,0.58,0.83
3,teologia_moral,361,279,0.61,0.80
4,catecismo,58,31,0.50,0.94
5,biblia,20,1,0.05,1.00
6,liturgia,158,82,0.46,0.89
7,gobierno_religioso,1122,1003,0.62,0.69
8,no_religioso,2262,2667,0.92,0.78


## 3. What does it say for the whole catalog?

---

## 3. ¿Qué dice para todo el catálogo?

In [7]:
# Celda 7
# Train on the whole sample, label the whole catalog and save both sets of answers:
classifier.fit(labeled["text"], labeled["llm_category"])
catalog["ml_category"] = classifier.predict(catalog["text"])
catalog[["source", "id", "ml_category"]].to_csv(CATALOG_PREDICTIONS, sep="\t", index=False)
labeled[["source", "id", "ml_category"]].to_csv(SAMPLE_PREDICTIONS, sep="\t", index=False)
catalog["ml_category"].value_counts()

no_religioso          98153
gobierno_religioso    34990
espiritual             9858
predicable             8373
historia_sacra         4093
teologia_moral         3678
liturgia               3470
catecismo               487
biblia                   67
Name: ml_category, dtype: int64